In [2]:
!pip install torch torchvision pandas tqdm scikit-learn matplotlib seaborn torchmetrics


  Using cached torchmetrics-1.8.2-py3-none-any.whl.metadata (22 kB)
  Using cached lightning_utilities-0.15.2-py3-none-any.whl.metadata (5.7 kB)
Using cached torchmetrics-1.8.2-py3-none-any.whl (983 kB)
Using cached lightning_utilities-0.15.2-py3-none-any.whl (29 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchmetrics] [torchmetrics]


In [ ]:
import torch, torchvision
print("torch:", torch.__version__, torch.__file__)
print("torchvision:", torchvision.__version__, torchvision.__file__)


torch: 2.10.0+cu128 /home/na1488tr-s/.local/lib/python3.10/site-packages/torch/__init__.py
torchvision: 0.25.0+cu128 /home/na1488tr-s/.local/lib/python3.10/site-packages/torchvision/__init__.py


In [17]:
import os 
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
import torchvision.transforms as T
from torchmetrics.classification import MulticlassRecall, MulticlassAccuracy
from tqdm import tqdm

from Data_Utility.lookup_size import lookup_size_from_excel
from Data_Utility.dataset import PollenFolderWithSizeDataset

from models.basemodel import CNNWithSizeMLP

print("All libraries imported successfully!")

All libraries imported successfully!


## Preparing data

In [18]:
train_dir = '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/Sorted_224_sizeTrain'
test_dir = '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/Sorted_224_sizeTest'
test_excel_path = '/home/na1488tr-s/Bachelor_Project_Statistics/Data/Size-data/Size_features/size_data.xlsx'

size_lookup, species_mean_lookup, global_mean = lookup_size_from_excel(test_excel_path)

classes = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
class_to_idx = {c: i for i, c in enumerate(classes)}

#train_tf = T.Compose([T.ToTensor()]) # converts PIL → Tensor
#test_tf = T.Compose([T.ToTensor()]) # converts PIL → Tensor

train_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


train_dataset = PollenFolderWithSizeDataset(img_dir=train_dir, class_to_idx=class_to_idx, size_lookup=size_lookup, species_mean_lookup=species_mean_lookup, global_mean=global_mean, transform=train_tf)
test_dataset = PollenFolderWithSizeDataset(img_dir=test_dir, class_to_idx=class_to_idx, size_lookup=size_lookup, species_mean_lookup=species_mean_lookup, global_mean=global_mean, transform=test_tf)




Total samples: 9844
No missing size data found. All rows have valid majoraxis and minoraxis values.


In [19]:
# Splitting the training dataset into training and validation sets
import random


val_ratio = 0.2
n_total = len(train_dataset)
n_val = int(n_total * val_ratio)
n_train = n_total - n_val

# for reproducibility
g = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(train_dataset, [n_train, n_val], generator=g)

print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")

Training samples: 6463, Validation samples: 1615, Test samples: 1922


## Base Model (Erik's)

In [20]:
from os import access

from sympy import Mul


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)
model = CNNWithSizeMLP(num_classes=len(classes)).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Performance metric
recall_metric = MulticlassRecall(num_classes=len(classes), average = None).to(device)
accuracy_metric = MulticlassAccuracy(num_classes=len(classes), average = "micro").to(device)  




Using device: cuda


In [21]:
def train_epoch(loader):
    model.train()
    total_loss = 0
    for imgs, sizes, labels in tqdm(loader):
        imgs = imgs.to(device)
        sizes = sizes.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs, sizes)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(loader):
    model.eval()
    
    recall_metric.reset()
    accuracy_metric.reset()
    
    with torch.no_grad():
        for imgs, sizes, labels in loader:
            imgs = imgs.to(device)
            sizes = sizes.to(device)
            labels = labels.to(device)

            outputs = model(imgs, sizes)
            preds = outputs.argmax(dim=1)
            
            recall_metric.update(preds, labels)
            accuracy_metric.update(preds, labels)
            
    recall_per_class = recall_metric.compute().cpu() # tensor of shape (num_classes,)
    accuracy = accuracy_metric.compute().cpu().item() # scalar
    return recall_per_class, accuracy

In [22]:
epochs_num = 25
best_val_acc = 0.0   # initialize before loop

for epoch in range(1, epochs_num + 1):
    train_loss = train_epoch(train_loader)
    val_recall_per_class, val_acc = eval_epoch(val_loader)
    
    print(f"Epoch {epoch}: loss {train_loss:.4f}, val_acc {val_acc:.4f}")
    print(f"Val Recall per class: {val_recall_per_class}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("Saved new best model!")

100%|██████████| 202/202 [00:18<00:00, 11.01it/s]


Epoch 1: loss 0.3016, val_acc 0.7133
Val Recall per class: tensor([0.6243, 0.7602, 0.8693, 0.6778, 0.8824, 0.5938, 0.0362, 0.5714, 1.0000,
        1.0000])
Saved new best model!


100%|██████████| 202/202 [00:18<00:00, 11.03it/s]


Epoch 2: loss 0.1490, val_acc 0.6873
Val Recall per class: tensor([0.6961, 0.9532, 0.9935, 1.0000, 0.6275, 0.3313, 0.3333, 0.7143, 0.2321,
        0.9064])


100%|██████████| 202/202 [00:18<00:00, 11.04it/s]


...

Epoch 23: loss 0.0115, val_acc 0.9604
Val Recall per class: tensor([1.0000, 0.7836, 1.0000, 0.9944, 1.0000, 0.9563, 0.9130, 0.9929, 0.9702,
        0.9942])


100%|██████████| 202/202 [00:18<00:00, 10.90it/s]


Epoch 24: loss 0.0168, val_acc 0.9932
Val Recall per class: tensor([1.0000, 0.9942, 0.9935, 0.9889, 0.9935, 0.9875, 1.0000, 0.9857, 0.9881,
        1.0000])
Saved new best model!


100%|██████████| 202/202 [00:18<00:00, 11.02it/s]


Epoch 25: loss 0.0118, val_acc 0.9424
Val Recall per class: tensor([0.9890, 1.0000, 0.9869, 0.6667, 0.9346, 0.9125, 0.9710, 0.9929, 1.0000,
        1.0000])


In [23]:
print("\nTraining complete.")
print("Loading best model for final test evaluation...")

model.load_state_dict(torch.load("best_model.pth"))

test_recall_per_class, test_acc = eval_epoch(test_loader)

print(f"\nFinal Test Accuracy: {test_acc:.4f}")
print(f"Final Test Recall per class: {test_recall_per_class}")


Training complete.
Loading best model for final test evaluation...

Final Test Accuracy: 0.7383
Final Test Recall per class: tensor([0.4366, 1.0000, 1.0000, 1.0000, 0.0000, 0.9100, 0.4450, 0.7605, 0.9157,
        1.0000])


In [ ]:
from collections import Counter

all_labels = [test_dataset[i][2].item() for i in range(len(test_dataset))]
counts = Counter(all_labels)

# show counts with class names
idx_to_class = {v: k for k, v in class_to_idx.items()}
for k in sorted(counts):
    print(k, idx_to_class[k], counts[k])


0 Bellis perennis 142
1 Brassica napus 200
2 Capsella bursa-pastoris 183
3 Cichorium intybus 130
4 Crepis capillaris 200
5 Hieracium umbellatum 200
6 Hypochaeris radicata 200
7 Sonchus arvensis 334
8 Tragopogon pratensis 166
9 Tussilago farfara 167
